# 1.2 Типы и коллекции: решения упражнений

Аналитика 101 · блок «Дата аналитика» · тема 01 «Python для анализа» · страница 1.2.

Сначала идут три блока кода со страницы: упражнения берут из них `data` и `orders`. Решения с разбором — во второй половине. Ноутбук выполнен целиком сверху вниз, все проверки прошли.

Загляните сюда после своей попытки: чужое решение понятнее, когда уже есть своё.

In [1]:
import platform

print("Python", platform.python_version())   # версия, на которой запущен ноутбук

Python 3.13.18


## Код страницы

### Разбор JSON-ответа API в словари и списки

In [2]:
import json

# Учебный ответ API интернет-магазина. В жизни он приходит по сети,
# здесь лежит строкой, чтобы пример работал без интернета
raw = """
{
  "shop": "учебный магазин",
  "orders": [
    {"id": 1041, "created": "2026-03-14T10:22:05", "brand": "Сова",
     "amount": 1200, "discount": 0.1, "paid": true, "promo": null},
    {"id": 1042, "created": "2026-03-14T18:03:41", "brand": "Кедр",
     "amount": 850, "discount": 0.0, "paid": true, "promo": "ВЕСНА"},
    {"id": 1043, "created": "2026-03-15T09:47:12", "brand": "Сова",
     "amount": 2300, "discount": 0.0, "paid": false, "promo": null},
    {"id": 1044, "created": "2026-03-28T21:15:00", "brand": "Луч",
     "amount": 640, "discount": 0.0, "paid": true, "promo": null},
    {"id": 1045, "created": "2026-04-02T08:30:59", "brand": "Сова",
     "amount": 1750, "discount": 0.1, "paid": true, "promo": "ВЕСНА"}
  ]
}
"""

data = json.loads(raw)              # строка JSON → словари и списки Python
print(type(data).__name__, list(data))

orders = data["orders"]             # по ключу "orders" лежит список
print(type(orders).__name__, len(orders))

first = orders[0]                   # каждый заказ — снова словарь
for key, value in first.items():
    print(f"{key:9}{type(value).__name__:9}{value!r}")

print(data["orders"][2]["brand"])   # путь вглубь: ключ → индекс → ключ

paid_total = 0                      # сумма оплаченных заказов
for o in orders:
    if o["paid"]:
        paid_total += o["amount"]
print("Оплачено:", paid_total, "₽")

dict ['shop', 'orders']
list 5
id       int      1041
created  str      '2026-03-14T10:22:05'
brand    str      'Сова'
amount   int      1200
discount float    0.1
paid     bool     True
promo    NoneType None
Сова
Оплачено: 4440 ₽


true стало True, null — None, 1200 — int, 0.1 — float. Заказ 1043 не оплачен, поэтому в сумму не попал.

### Частоты брендов: dict, Counter и set

In [3]:
from collections import Counter

brands = []                         # бренды по порядку заказов, с повторами
for o in orders:
    brands.append(o["brand"])
print(brands)

counts = {}                         # частоты вручную: бренд → число заказов
for b in brands:
    counts[b] = counts.get(b, 0) + 1
print(counts)

c = Counter(brands)                 # то же самое одной строкой
print(c)
print(c.most_common(1))

unique = set(brands)                # уникальные бренды
print(len(brands), len(unique), sorted(unique))
print("Луч" in unique, "Лис" in unique)

['Сова', 'Кедр', 'Сова', 'Луч', 'Сова']
{'Сова': 3, 'Кедр': 1, 'Луч': 1}
Counter({'Сова': 3, 'Кедр': 1, 'Луч': 1})
[('Сова', 3)]
5 3 ['Кедр', 'Луч', 'Сова']
True False


Вывод одинаков от запуска к запуску: множество печатается через sorted, а словарь и Counter хранят порядок добавления ключей.

### Срезы строки с датой и выручка по месяцам

In [4]:
created = orders[0]["created"]
print(created)
print(created[:4], created[5:7], created[8:10])   # год, месяц, день
print(created[:10], created[-8:])                 # дата и время
print(created[:7])                                # год и месяц
print(created.split("T"))                         # метод строки: разрезать по T

by_month = {}                       # выручка по месяцам
for o in orders:
    month = o["created"][:7]
    by_month[month] = by_month.get(month, 0) + o["amount"]
print(by_month)

2026-03-14T10:22:05
2026 03 14
2026-03-14 10:22:05
2026-03
['2026-03-14', '10:22:05']
{'2026-03': 4990, '2026-04': 1750}


Здесь учтены все заказы, включая неоплаченный 1043. Срез работает, пока все даты записаны в одном формате с ведущими нулями.

## Упражнения

Данные те же: список `orders` из пяти заказов и словарь `data` из первого блока.

### Упражнение 1. Срезы: час заказа

В строке `created` час стоит на позициях 11 и 12: `"2026-03-14T10:22:05"`. Соберите список `hours` — час каждого заказа целым числом, в порядке заказов.

In [5]:
hours = []
for o in orders:
    hours.append(int(o["created"][11:13]))   # срез [11:13] — два символа часа; int() делает из строки число
print(hours)

[10, 18, 9, 21, 8]


In [6]:
assert hours == [10, 18, 9, 21, 8], f"Ожидался список [10, 18, 9, 21, 8], получено {hours!r}"
print("Упражнение 1: верно")

Упражнение 1: верно


**Разбор.** Срез `[11:13]` берёт символы с индексами 11 и 12: правая граница в срез не входит. Без `int()` в списке остались бы строки `'10'`, `'18'`, а строки сравниваются по символам: `'9' > '10'`.

### Упражнение 2. Частоты: заказы по промокодам

Посчитайте, сколько заказов пришло с каждым промокодом. Результат — словарь `promo_counts`: промокод → число заказов. Заказы без промокода попадают под ключ `None`.

In [7]:
promo_counts = {}
for o in orders:
    p = o["promo"]
    promo_counts[p] = promo_counts.get(p, 0) + 1   # .get(p, 0): ноль, если ключа ещё нет
print(promo_counts)

promos = []                                        # то же через Counter
for o in orders:
    promos.append(o["promo"])
print(Counter(promos))

{None: 3, 'ВЕСНА': 2}
Counter({None: 3, 'ВЕСНА': 2})


In [8]:
assert promo_counts is not ..., "Замените ... своим кодом"
assert dict(promo_counts) == {None: 3, "ВЕСНА": 2}, f"Ожидалось {{None: 3, 'ВЕСНА': 2}}, получено {promo_counts!r}"
print("Упражнение 2: верно")

Упражнение 2: верно


**Разбор.** `None` годится в ключи словаря: он неизменяем. Поэтому отдельная ветка «если промокода нет» не нужна. `Counter` — тот же словарь с готовым подсчётом, и проверка через `==` для него работает так же.

### Упражнение 3. Множества: кто не вернулся в апреле

Соберите два множества: `march` — бренды из заказов марта 2026 года, `april` — из заказов апреля. Месяц берите срезом `created[:7]`. Затем составьте список `lost`: мартовские бренды, которых нет в апреле, по алфавиту.

In [9]:
march, april = set(), set()          # пустое множество — set(), а не {}: {} — это пустой словарь
for o in orders:
    month = o["created"][:7]
    if month == "2026-03":
        march.add(o["brand"])        # .add() кладёт элемент во множество; повтор ничего не меняет
    elif month == "2026-04":
        april.add(o["brand"])

lost = []
for b in sorted(march):              # sorted даёт алфавитный порядок: у самого множества порядка нет
    if b not in april:
        lost.append(b)
print(sorted(march), sorted(april), lost)

['Кедр', 'Луч', 'Сова'] ['Сова'] ['Кедр', 'Луч']


In [10]:
assert march == {"Сова", "Кедр", "Луч"}, f"В марте покупали три бренда, получено {march!r}"
assert april == {"Сова"}, f"В апреле покупали один бренд, получено {april!r}"
assert lost == ["Кедр", "Луч"], f"Ожидался список ['Кедр', 'Луч'], получено {lost!r}"
print("Упражнение 3: верно")

Упражнение 3: верно


**Разбор.** Цикл с `not in` можно заменить одной операцией — разностью множеств: `sorted(march - april)`. На странице её нет, но в работе она встречается постоянно: «кто был и пропал», «какие товары есть в одной выгрузке и нет в другой».

### Упражнение 4. Копия: функция, которая портит исходник

Функция `drop_unpaid` удаляет неоплаченные заказы прямо из переданного списка. Получите `paid_orders` — список из четырёх оплаченных заказов — так, чтобы в `orders` остались все пять.

Если проверка скажет, что `orders` стал короче, выполните ноутбук заново с первой ячейки: исходные данные уже испорчены.

In [11]:
def drop_unpaid(rows):
    """Удаляет неоплаченные заказы из списка rows и возвращает его же."""
    for o in rows[:]:            # идём по копии, удаляем из самого списка
        if not o["paid"]:
            rows.remove(o)
    return rows

In [12]:
paid_orders = drop_unpaid(orders.copy())   # функция получает копию и удаляет заказы из неё
print(len(orders), len(paid_orders), paid_orders is orders)

5 4 False


In [13]:
assert len(orders) == 5, "orders стал короче: функция получила сам список, а не копию. Выполните ноутбук заново с первой ячейки"
assert isinstance(paid_orders, list), "paid_orders должен быть списком"
assert len(paid_orders) == 4 and all(o["paid"] for o in paid_orders), "В paid_orders должны остаться четыре оплаченных заказа"
print("Упражнение 4: верно")

Упражнение 4: верно


**Разбор.** `orders.copy()` — поверхностная копия: новый список с теми же словарями-заказами. Здесь её хватает, потому что функция меняет состав списка, а сами заказы не трогает. Когда меняют то, что лежит внутри, поверхностной копии мало — это упражнение 5.

### Упражнение 5. Вложенная копия: черновик ответа API

Сделайте `draft` — независимую копию словаря `data`. Ячейка проверки обнулит в черновике сумму первого заказа, а в исходном `data` она должна остаться 1200.

Если проверка скажет, что сумма изменилась и в `data`, выполните ноутбук заново с первой ячейки.

In [14]:
import copy

draft = copy.deepcopy(data)   # копируется вся структура: словарь, список заказов и каждый заказ
print(draft is data, draft["orders"] is data["orders"], draft == data)

False False True


In [15]:
assert isinstance(draft, dict) and draft is not data, "draft должен быть отдельным словарём, а не вторым именем data"
draft["orders"][0]["amount"] = 0   # правим черновик
assert data["orders"][0]["amount"] == 1200, "Сумма изменилась и в data: копия поверхностная, заказы внутри общие. Выполните ноутбук заново и сделайте глубокую копию"
print("Упражнение 5: верно")

Упражнение 5: верно


**Разбор.** `data.copy()` и `dict(data)` копируют только верхний словарь. По ключу `"orders"` в такой копии лежит тот же список, что в оригинале, поэтому правка первого заказа видна через оба имени. `copy.deepcopy` проходит структуру до дна и копирует каждый вложенный объект.